# Conquer Mobile → TripoSR | تشغيل كامل مُراجع\nخلية تشغيل واحدة: تفحص T4 وCUDA، تثبّت torchmcubes بالطريقة الصحيحة دون build isolation، تختبر CUDA، تنشئ أول مجسم وتتحقق من OBJ/GLB، ثم تكمل الصور الـ42 تلقائياً. لا تحتاج HF Secret ولا أي خدمة مدفوعة.\n\n**مهم:** اختر T4 GPU ثم شغّل خلية الكود الوحيدة. لا تضغط Run all؛ اضغط تشغيل الخلية الكبيرة مرة واحدة واتركها حتى تنتهي.

In [ ]:
#@title تشغيل كامل: تثبيت مضبوط + أول مجسم + بقية الصور الـ42
import os, sys, json, shutil, subprocess, pathlib, struct, gc, csv, time
from pathlib import Path

def run(cmd, **kwargs):
    print("\\n>>>", " ".join(map(str, cmd))[:260], flush=True)
    subprocess.run(cmd, check=True, **kwargs)

# 1) تأكيد بيئة T4 ووجود CUDA Toolkit قبل أي تثبيت
import torch
assert torch.cuda.is_available(), "اتصل ببيئة T4 GPU أولاً من Runtime > Change runtime type."
gpu = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| GPU:", gpu, "| Capability:", cap, flush=True)
assert cap == (7, 5), f"هذه الخلية مضبوطة على T4 (compute 7.5)، لكن الجهاز الحالي هو {gpu} / {cap}."
nvcc = shutil.which("nvcc")
assert nvcc, "لم أجد nvcc؛ جلسة GPU لا تحتوي CUDA Toolkit اللازمة لبناء torchmcubes. أوقفت التشغيل قبل التثبيت."
nvcc_text = subprocess.check_output([nvcc, "--version"], text=True)
print(nvcc_text.splitlines()[-1], flush=True)
nvcc_match = __import__("re").search(r"release (\\d+)\\.", nvcc_text)
torch_cuda_major = int(torch.version.cuda.split(".")[0])
assert nvcc_match and int(nvcc_match.group(1)) == torch_cuda_major, f"اختلاف إصدار CUDA: PyTorch={torch.version.cuda}, nvcc={nvcc_text.splitlines()[-1]}"

# 2) جلب المستودعين؛ رابط المستودع ذو النقطتين قبل git مؤكد من GitHub API
ASSET_REPO = Path("/content/Conquer-Mobile-Assets")
TSR_REPO = Path("/content/TripoSR")
if not ASSET_REPO.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/ZzwahwahzZ/Conquer-Mobile-Assets..git", str(ASSET_REPO)])
if not TSR_REPO.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/VAST-AI-Research/TripoSR.git", str(TSR_REPO)])

IMAGES = ASSET_REPO / "original_images"
image_files = sorted(p for p in IMAGES.rglob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp"))
assert len(image_files) == 42, f"المتوقع 42 صورة أصلية، الموجود {len(image_files)}."
characters = sorted(p for p in (IMAGES / "characters").rglob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp"))
assert characters, "لم أجد صور الشخصيات الأصلية."
first_image = characters[0]
print(f"تم العثور على {len(image_files)} صورة أصلية. أول اختبار: {first_image.relative_to(IMAGES)}", flush=True)

# 3) تثبيت متطلبات TripoSR دون إدخال torchmcubes في build isolation
#    السبب: torchmcubes يحتاج رؤية PyTorch المثبت مسبقاً أثناء بناء CMake.
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel",
     "scikit-build-core>=1.0", "cmake>=3.24", "ninja", "pybind11>=2.10"])
run([sys.executable, "-m", "pip", "install",
     "omegaconf==2.3.0", "Pillow==10.1.0", "einops==0.7.0",
     "transformers==4.35.0", "trimesh==4.0.5", "rembg",
     "huggingface-hub==0.25.0", "imageio"])

# 4) بناء torchmcubes مع PyTorch وCUDA الخاصين بجلسة Colab نفسها
import torch.utils
torch_prefix = torch.utils.cmake_prefix_path
os.environ["CMAKE_PREFIX_PATH"] = torch_prefix + os.pathsep + os.environ.get("CMAKE_PREFIX_PATH", "")
os.environ["CMAKE_ARGS"] = f"-DCMAKE_PREFIX_PATH={torch_prefix} -DCMAKE_CUDA_ARCHITECTURES=75"
os.environ["CMAKE_BUILD_PARALLEL_LEVEL"] = "2"
run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps",
     "git+https://github.com/tatsy/torchmcubes.git"])

import torchmcubes
print("torchmcubes:", torchmcubes.__version__, "| CUDA extension:", torchmcubes.HAS_CUDA, flush=True)
assert torchmcubes.HAS_CUDA, "تم تثبيت torchmcubes لكن نسخة CUDA غير موجودة؛ أوقفت العمل حتى لا يتحول الاستخراج إلى CPU بطيء."
# اختبار واحد حاسم لامتداد CUDA قبل تحميل النموذج
probe = torch.zeros((12, 12, 12), device="cuda")
probe[3:9, 3:9, 3:9] = 1
probe_v, probe_f = torchmcubes.marching_cubes(probe, 0.5)
assert probe_v.is_cuda and probe_f.is_cuda and probe_v.numel() > 0 and probe_f.numel() > 0
del probe, probe_v, probe_f
torch.cuda.empty_cache()
print("نجح اختبار marching-cubes على T4.", flush=True)

# 5) تحميل TripoSR مرة واحدة فقط، ثم إعادة استخدامه لكل الصور
os.chdir(TSR_REPO)
from PIL import Image
import numpy as np
import trimesh
from rembg import new_session, remove
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground

device = "cuda:0"
print("تحميل أوزان TripoSR الرسمية من Hugging Face...", flush=True)
model = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
model.renderer.set_chunk_size(4096)
model.to(device)
model.eval()
rembg_session = new_session()
print("النموذج جاهز. سيُنفذ اختبار الشخصية الأولى ثم يكمل بقية الصور تلقائياً.", flush=True)

OUT = Path("/content/Conquer_TripoSR_42_Models")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)
results = []

def prepare_image(path):
    im = Image.open(path)
    if im.mode not in ("RGB", "RGBA"):
        im = im.convert("RGBA")
    im = remove_background(im, rembg_session)
    im = resize_foreground(im, 0.85)
    arr = np.asarray(im).astype(np.float32) / 255.0
    if arr.ndim != 3 or arr.shape[-1] != 4:
        raise RuntimeError(f"خلفية الصورة لم تُجهّز بصيغة RGBA: {path.name}")
    alpha = arr[:, :, 3:4]
    if not np.any(alpha > 0):
        raise RuntimeError(f"الصورة أصبحت شفافة بالكامل بعد إزالة الخلفية: {path.name}")
    rgb = arr[:, :, :3] * alpha + (1.0 - alpha) * 0.5
    return Image.fromarray(np.clip(rgb * 255.0, 0, 255).astype(np.uint8))

def export_and_verify(mesh, folder, stem):
    if len(mesh.vertices) == 0 or len(mesh.faces) == 0:
        raise RuntimeError("النموذج الناتج فارغ.")
    # extract_mesh(..., False) يمنع ألوان الرؤوس المستخرجة من صورة الإدخال.
    obj_path = folder / f"{stem}.obj"
    glb_path = folder / f"{stem}.glb"
    obj_data = mesh.export(file_type="obj")
    if isinstance(obj_data, str):
        obj_path.write_text(obj_data, encoding="utf-8")
    else:
        obj_path.write_bytes(obj_data)
    glb_data = mesh.export(file_type="glb")
    if not isinstance(glb_data, (bytes, bytearray)):
        raise RuntimeError("تصدير GLB لم يُرجع بيانات ثنائية.")
    glb_path.write_bytes(glb_data)
    assert obj_path.stat().st_size > 1000 and glb_path.stat().st_size > 1000, "ملف تصدير صغير أو فارغ."
    # فحص GLB مباشرة: لا صور/خامات نسيجية ولا COLOR_0 (ألوان رؤوس).
    data = glb_path.read_bytes()
    magic, version, total_len = struct.unpack_from("<4sII", data, 0)
    assert magic == b"glTF" and version == 2 and total_len == len(data), "ترويسة GLB غير سليمة."
    json_len, json_type = struct.unpack_from("<I4s", data, 12)
    assert json_type == b"JSON", "لم أجد JSON chunk في GLB."
    gltf = json.loads(data[20:20+json_len].decode("utf-8").rstrip(" \\x00"))
    assert not gltf.get("images") and not gltf.get("textures"), "ظهر نسيج/صورة داخل GLB رغم طلب مجسم بلا خامات."
    for m in gltf.get("meshes", []):
        for prim in m.get("primitives", []):
            assert "COLOR_0" not in prim.get("attributes", {}), "ظهر vertex color داخل GLB."
    # إعادة فتح الملفين للتأكد من سلامة الهندسة بعد التصدير
    check_obj = trimesh.load(obj_path, force="mesh", process=False)
    check_glb = trimesh.load(glb_path, force="mesh", process=False)
    assert len(check_obj.vertices) and len(check_obj.faces) and len(check_glb.vertices) and len(check_glb.faces)
    return obj_path, glb_path, len(mesh.vertices), len(mesh.faces)

def make_one(path, first=False):
    rel = path.relative_to(IMAGES)
    folder = OUT / rel.parent
    folder.mkdir(parents=True, exist_ok=True)
    safe = path.stem.replace(" ", "_")
    print(f"إنشاء المجسم: {rel}", flush=True)
    pil = prepare_image(path)
    with torch.inference_mode():
        scene_codes = model([pil], device=device)
        mesh = model.extract_mesh(scene_codes, False, resolution=256)[0]
    paths = export_and_verify(mesh, folder, safe)
    del pil, scene_codes, mesh
    gc.collect()
    torch.cuda.empty_cache()
    return {"source": str(rel), "status": "OK", "obj": str(paths[0].relative_to(OUT)),
            "glb": str(paths[1].relative_to(OUT)), "vertices": paths[2], "faces": paths[3]}

# 6) أول شخصية اختبار حقيقي. إذا نجحت كل الفحوص يبدأ الباقي تلقائياً.
first_result = make_one(first_image, first=True)
results.append(first_result)
print("\\nنجاح أول مجسم OBJ + GLB والتحقق من خلو GLB من الخامات والصبغات المستخرجة. استكمال بقية الصور...", flush=True)

remaining = [p for p in image_files if p != first_image]
for index, path in enumerate(remaining, start=2):
    try:
        results.append(make_one(path))
        print(f"اكتمل {index}/42", flush=True)
    except Exception as e:
        print(f"تعذر إنشاء {path.relative_to(IMAGES)}: {type(e).__name__}: {e}", flush=True)
        results.append({"source": str(path.relative_to(IMAGES)), "status": "FAILED", "error": f"{type(e).__name__}: {e}"})
        gc.collect()
        torch.cuda.empty_cache()

# 7) تقرير + ZIP بالمخرجات التي اكتملت
report = OUT / "generation_report.csv"
with report.open("w", newline="", encoding="utf-8-sig") as f:
    fields = sorted({k for row in results for k in row.keys()})
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(results)

ok = [r for r in results if r["status"] == "OK"]
failed = [r for r in results if r["status"] != "OK"]
assert ok, "لم ينتج أي مجسم صالح."
zip_path = Path("/content/Conquer_TripoSR_42_Models.zip")
with __import__("zipfile").ZipFile(zip_path, "w", __import__("zipfile").ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(OUT))
print("\\n========== النتيجة ==========", flush=True)
print(f"نجح: {len(ok)}/42 | تعذر: {len(failed)} | OBJ: {len(ok)} | GLB: {len(ok)}", flush=True)
print("الأرشيف:", zip_path, "| الحجم:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB", flush=True)
if failed:
    print("تفاصيل الصور المتعثرة موجودة في generation_report.csv داخل الأرشيف.", flush=True)
from google.colab import files
files.download(str(zip_path))

